# MLP for the PhiUSIIL Phishing Dataset

This notebook trains the neural-network baseline for the phishing dataset. The MLP is placed inside a pipeline with StandardScaler, and hyperparameters are tuned using cross-validation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, matthews_corrcoef,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay,
)

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "PhiUSIIL_Phishing_URL_Dataset.csv"
OUTPUT_DIR = BASE_DIR / "outputs" / "mlp_phiusiil"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "label"
PHISHING_LABEL = 0
SAMPLES_PER_CLASS = 5000

print("Dataset found:", DATA_PATH.exists())
if not DATA_PATH.exists():
    raise FileNotFoundError("Place PhiUSIIL_Phishing_URL_Dataset.csv in this folder.")

## Load data and prepare numeric features

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False).dropna(subset=[TARGET_COL]).copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)
df["is_phishing"] = (df[TARGET_COL] == PHISHING_LABEL).astype(int)

text_or_id_cols = ["FILENAME", "URL", "Domain", "Title", "TLD"]
features = [
    c for c in df.columns
    if c not in text_or_id_cols + [TARGET_COL, "is_phishing"]
    and pd.api.types.is_numeric_dtype(df[c])
]
for c in features:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df[c].fillna(df[c].median())

print("Feature count:", len(features))
display(df["is_phishing"].value_counts())

## Balanced sample and train/test split

In [ ]:
df_pos = df[df["is_phishing"] == 1]
df_neg = df[df["is_phishing"] == 0]
n_per_class = min(SAMPLES_PER_CLASS, len(df_pos), len(df_neg))
df_model = pd.concat([
    df_pos.sample(n=n_per_class, random_state=42),
    df_neg.sample(n=n_per_class, random_state=42),
], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

X = df_model[features]
y = df_model["is_phishing"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)

pd.DataFrame({"feature": features}).to_csv(OUTPUT_DIR / "mlp_feature_list.csv", index=False)
print("Training rows:", X_train.shape[0], "Test rows:", X_test.shape[0])

## Tune and train the MLP

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
model = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(max_iter=300, early_stopping=True, random_state=42)),
])
param_grid = {
    "mlp__hidden_layer_sizes": [(50,), (100,), (50, 25)],
    "mlp__alpha": [0.0001, 0.001],
    "mlp__learning_rate_init": [0.001],
}

grid = GridSearchCV(model, param_grid=param_grid, scoring="f1", cv=cv, n_jobs=-1, verbose=1, return_train_score=True)
grid.fit(X_train, y_train)
pd.DataFrame(grid.cv_results_).to_csv(OUTPUT_DIR / "mlp_gridsearch_results.csv", index=False)

print("Best parameters:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

## Final evaluation on the held-out test set

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_score = best_model.predict_proba(X_test)[:, 1]
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) else np.nan

metrics = {
    "model": "MLP",
    "best_params": str(grid.best_params_),
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall_sensitivity": recall_score(y_test, y_pred, zero_division=0),
    "specificity": specificity,
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_score),
    "average_precision_pr_auc": average_precision_score(y_test, y_score),
    "mcc": matthews_corrcoef(y_test, y_pred),
    "tn": tn, "fp": fp, "fn": fn, "tp": tp,
}

pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / "mlp_phiusiil_results.csv", index=False)
print(classification_report(y_test, y_pred, target_names=["legitimate", "phishing"]))
display(pd.DataFrame([metrics]))

## Report-ready plots

In [ ]:
ConfusionMatrixDisplay(cm, display_labels=["legitimate", "phishing"]).plot(values_format="d")
plt.title("MLP Confusion Matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mlp_confusion_matrix.png", dpi=300)
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_score)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {metrics['roc_auc']:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("MLP ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mlp_roc_curve.png", dpi=300)
plt.show()

precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_score)
plt.figure(figsize=(7, 5))
plt.plot(recall_curve, precision_curve, label=f"AP = {metrics['average_precision_pr_auc']:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("MLP Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mlp_precision_recall_curve.png", dpi=300)
plt.show()

print("MLP outputs saved to:", OUTPUT_DIR)